In [1]:
#임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
#데이터 로드
data = pd.read_csv("../risk_data/Sleep Health and Lifestyle Dataset.csv")

In [3]:
#사용할 피처
sleep_efficiency_features = [
    "Age",
    "Gender",
    "BMI",
    "Sleep_Duration",
    "Stress_Level",
    "Daytime_Sleepiness",
    "Caffeine_Intake_mg",
    "Physical_Activity_Minutes",
    "Screen_Time_Hours",
    "Smoking_Status",
    "Alcohol_Consumption"
]

In [4]:
#넘버 피처
numeric_corr_features = [
    "Age",
    "BMI",
    "Sleep_Duration",
    "Stress_Level",
    "Daytime_Sleepiness",
    "Caffeine_Intake_mg",
    "Physical_Activity_Minutes",
    "Screen_Time_Hours",
    "Sleep_Efficiency"
]

In [5]:
#상관관계 분석
sleep_efficiency_corr = data[numeric_corr_features].corr()["Sleep_Efficiency"]
sleep_efficiency_corr = sleep_efficiency_corr.drop("Sleep_Efficiency")
sleep_efficiency_corr = sleep_efficiency_corr.reindex(
    sleep_efficiency_corr.abs().sort_values(ascending=False).index
)
sleep_efficiency_corr

Daytime_Sleepiness          -0.781395
Stress_Level                -0.764288
Sleep_Duration               0.561132
BMI                         -0.180211
Screen_Time_Hours            0.007600
Physical_Activity_Minutes    0.005884
Caffeine_Intake_mg          -0.001377
Age                         -0.000794
Name: Sleep_Efficiency, dtype: float64

In [6]:
#범주형 포함
corr_data = data[
    sleep_efficiency_features + ["Sleep_Efficiency"]
].copy()
#성별
corr_data["Gender"] = corr_data["Gender"].map({
    "Female": 0,
    "Male": 1
})

#담배
corr_data["Smoking_Status"] = corr_data["Smoking_Status"].map({
    "No": 0,
    "Yes": 1
})

#알콜
corr_data["Alcohol_Consumption"] = corr_data["Alcohol_Consumption"].map({
    "No": 0,
    "Yes": 1
})

all_corr = corr_data.corr()["Sleep_Efficiency"]
all_corr = all_corr.drop("Sleep_Efficiency")
all_corr = all_corr.reindex(
    all_corr.abs().sort_values(ascending=False).index
)
all_corr

Daytime_Sleepiness          -0.781395
Stress_Level                -0.764288
Sleep_Duration               0.561132
BMI                         -0.180211
Alcohol_Consumption         -0.008185
Screen_Time_Hours            0.007600
Physical_Activity_Minutes    0.005884
Gender                       0.004266
Caffeine_Intake_mg          -0.001377
Smoking_Status              -0.000889
Age                         -0.000794
Name: Sleep_Efficiency, dtype: float64

### 전처리 시작

In [7]:

data["Stress_Level"] = data["Stress_Level"].map({
    4: "Low",
    7: "Medium",
    9: "High"
})

In [8]:
X = data[sleep_efficiency_features]
y = data["Sleep_Efficiency"]

In [9]:
X.head()

,Age,Gender,BMI,Sleep_Duration,Stress_Level,Daytime_Sleepiness,Caffeine_Intake_mg,Physical_Activity_Minutes,Screen_Time_Hours,Smoking_Status,Alcohol_Consumption
0,49,Female,31.6,5.66,Medium,5,148,38.7,3.2,Yes,No
1,40,Female,28.7,6.93,Low,1,138,55.9,8.2,No,No
2,42,Female,24.4,7.18,Low,2,195,53.3,3.9,No,No
3,51,Male,22.8,7.77,Low,2,183,47.6,7.1,No,No
4,44,Female,25.4,7.23,Low,2,205,87.1,6.3,No,No


In [10]:
# 숫자형 , 범주형 
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

In [11]:
numeric_features

['Age',
 'BMI',
 'Sleep_Duration',
 'Daytime_Sleepiness',
 'Caffeine_Intake_mg',
 'Physical_Activity_Minutes',
 'Screen_Time_Hours']

In [12]:
categorical_features

['Gender', 'Stress_Level', 'Smoking_Status', 'Alcohol_Consumption']

In [13]:
#데이터셋 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
X_train = X_train.copy()
X_test = X_test.copy()

In [15]:
#카페인 이상치 처리
outlier_features = [
    "Caffeine_Intake_mg",
    "Physical_Activity_Minutes"
]

In [16]:
outlier_bounds = {}

In [17]:
for column in outlier_features:
    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = max(0, Q1 - 1.5 * IQR)
    upper_bound = Q3 + 1.5 * IQR

    outlier_bounds[column] = (
        lower_bound,
        upper_bound
    )

    X_train[column] = X_train[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    X_test[column] = X_test[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

In [18]:
#스케일러
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

In [19]:
#인코더 
encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

In [20]:
# 데이터 결합
X_train_final = np.hstack([X_train[numeric_features].values,X_train_encoded])
X_test_final = np.hstack([X_test[numeric_features].values,X_test_encoded])

In [21]:
X_train_final.shape, X_test_final.shape

((24000, 16), (6000, 16))

In [22]:
data["Stress_Level"].value_counts(dropna=False)

Stress_Level
Low       12026
Medium    11207
High       6767
Name: count, dtype: int64

In [23]:
#데이터 
preprocessing_data_sleep_efficiency = {
    "X_train": X_train_final,
    "X_test": X_test_final,
    "y_train": y_train,
    "y_test": y_test,
    "scaler": scaler,
    "encoder": encoder,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "outlier_bounds": outlier_bounds
}

In [27]:
joblib.dump(
    preprocessing_data_sleep_efficiency,
    "../risk_data/efficiency_prep_yt_ver.pkl"
)
print("Sleep Efficiency 전처리 데이터 저장 완료")

Sleep Efficiency 전처리 데이터 저장 완료
